# Prétraitement & Validation Pipeline — Notebook 02

## Objectif
Valider le pipeline de chargement, de fenêtrage et de visualisation des données CT.
Ce notebook teste le module `O1_preprocessing` sur un ou plusieurs patients et vérifie que les volumes prétraités sont conformes aux attentes.

## Prérequis
- **Librairies** : `numpy`, `matplotlib`, `pydicom`, `scikit-image`
- **Données** : `data/raw/` (DICOM) ou `data/processed/` (volumes `.npy`)
- **Modules** : `src.O1_preprocessing.loader`, `src.O1_preprocessing.split_data`

## Étapes principales
1. [Imports & configuration](#1-imports--configuration)
2. [Découpage train/val/test](#2-d%C3%A9coupage-trainvaltest)
3. [Chargement et fenêtrage](#3-chargement-et-fen%C3%Aatrage)
4. [Visualisation comparative](#4-visualisation-comparative)
5. [Vérification du masque GT](#5-v%C3%A9rification-du-masque-gt)

> **Lien avec le notebook précédent :** Ce notebook fait suite à `01_exploration.ipynb` et se concentre sur la validation du prétraitement plutôt que sur l'exploration visuelle libre.


## 1. Imports & configuration

### 1.1 Chargement des modules et configuration du chemin racine

Import des modules du projet (`loader`, `split_data`) et configuration du chemin racine.
Ce bloc est commun à tous les notebooks et doit être exécuté en premier.

> **Sortie :** modules `src/` accessibles dans tout le notebook.


In [ ]:
import sys
import os

# On remonte d'un cran pour atteindre la racine du projet
root_path = os.path.abspath(os.path.join('..'))
if root_path not in sys.path:
    sys.path.append(root_path)

from src.O1_preprocessing.split_data import create_split
from src.O1_preprocessing.loader import get_available_patients

## 2. Découpage train/val/test

### 2.1 Création ou chargement du split patient

Génération d'un split reproductible des patients en ensembles train / validation / test.
La graine aléatoire (`seed`) garantit la reproductibilité des résultats d'une exécution à l'autre.

> **Sortie :** fichier `data/splits.json` contenant les listes de patients par ensemble.


In [ ]:
# ═══════════════════════════════════════════════════
# Vérification fenêtre HU active (doit être [-600, 400])
# ═══════════════════════════════════════════════════
from src.O1_preprocessing.normalisation import HU_WINDOWS
lung_win = HU_WINDOWS['lung']
print('Fenêtres HU configurées :')
for name, win in HU_WINDOWS.items():
    tag = ' ← active EP' if name == 'lung' else ''
    print(f'  {name:15s} : {win}{tag}')
assert lung_win == (-600, 400), f'Fenêtre incorrecte : {lung_win}'
print(f'Assertion OK : {lung_win}')


### 2.2 Inventaire des patients disponibles

Recensement des patients ayant à la fois un volume CT et un masque GT valides.
Distingue les patients avec données complètes de ceux avec données partielles.

> **Sortie :** `raw_inventory` — dictionnaire avec clés `with_both`, `vol_only`, `mask_only`.


In [ ]:
# from src.O1_preprocessing.split_data import create_split

# # Utilisation du loader pour avoir la liste des patients valides
# from src.O1_preprocessing.loader import get_available_patients

inventory = get_available_patients("../data")
valid_patients = inventory['with_both']  # ['PAT001', 'PAT002', ...]

# Créeation du split sur ces patients
from src.O1_preprocessing.split_data import random_split
import json
from pathlib import Path

split_dict = random_split(
    patients=valid_patients,
    train_ratio=0.7,
    val_ratio=0.15,
    test_ratio=0.15,
    seed=42
)

# Ajout des métadonnées et sauvegarder
split_dict["metadata"] = {
    "total_patients": len(valid_patients),
    "seed": 42,
    "ratios": {"train": 0.7, "val": 0.15, "test": 0.15},
    "note": "Split basé sur patients avec volume + masque validés"
}

output_path = Path("../data/splits/train_val_test.json")
output_path.parent.mkdir(parents=True, exist_ok=True)
with open(output_path, 'w') as f:
    json.dump(split_dict, f, indent=4)

print(f"✅ Split réel sauvegardé : {len(split_dict['train'])} Train, {len(split_dict['val'])} Val, {len(split_dict['test'])} Test")

## 3. Chargement et fenêtrage

### 3.1 Sélection du patient de test et chargement

Sélection du premier patient disponible et chargement de son volume CT en mémoire.
Le mode `force_raw=False` charge la version prétraitée (normalisée [0,1]) si disponible.

> **Sortie :** `volume` (ndarray 3D), `mask_gt` (ndarray binaire), `meta` (métadonnées).


In [ ]:
import numpy as np
from pathlib import Path
from src.O1_preprocessing.loader import get_available_patients, load_patient_data

# from src.O1_preprocessing.processor import resample, normalize

# 1. Configuration des chemins
input_dir = Path("../data")
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

# 2. Récupération de l'inventaire via loader
inventory = get_available_patients(input_dir)

for p_id in inventory['with_both']:
    print(f"🏗️  Preprocessing : {p_id}")
    
    # A. Chargement 
    vol, mask, meta = load_patient_data(p_id, inventory)
    
    # B. Application du fenêtrage Pulmonaire (-1000 à 400 HU)
    # ⚠️ load_patient_data charge déjà les .npy normalisés si use_processed=True
    if not inventory.get('use_processed', False):
        vol_proc = np.clip(vol, -1000, 400)
        vol_proc = (vol_proc - (-1000)) / (400 - (-1000))
    else:
        # Volume déjà normalisé [0,1], on le garde tel quel
        vol_proc = vol
    
    # C. Sauvegarde au format .npy (ultra rapide pour l'IA)
    patient_path = output_dir / p_id
    patient_path.mkdir(exist_ok=True)
    
    np.save(patient_path / "volume.npy", vol_proc.astype(np.float32))
    np.save(patient_path / "mask.npy", mask.astype(np.uint8))
    
    print(f"✅ Sauvegardé dans processed/{p_id}/")

### 3.2 Vérification des dimensions et de la plage de valeurs

Contrôle que le volume chargé possède les bonnes dimensions et que ses valeurs sont dans la plage attendue.
Ce contrôle détecte les problèmes de normalisation ou d'alignement volume/masque.


In [ ]:

import sys
import os
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import seaborn as sns

# Configuration du style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)

# Ajout du chemin vers 'src'
sys.path.append(os.path.abspath('../'))

# Importation des modules
try:
    from src.O1_preprocessing.loader import load_dicom_volume, load_mask, get_available_patients
    from src.O1_preprocessing.normalisation import apply_windowing, HU_WINDOWS, convert_to_hounsfield
    print("✅ Importation réussie !")
except ImportError as e:
    print(f"❌ Erreur d'importation: {e}")
    sys.exit(1)

# Création des dossiers pour sauvegarder les résultats
os.makedirs("../figures/preprocessing", exist_ok=True)


### 3.3 Application des fenêtrages HU

Génération des vues diagnostiques via les fenêtrages HU standard :
- **Pulmonaire** [-600, 400] : parenchyme et vaisseaux pulmonaires.
- **Médiastin** [-150, 250] : structures vasculaires centrales et cœur.


In [ ]:
# ===================================
# PARTIE 1: ANALYSE D'UN PATIENT 
# ===================================

print("="*60)
print("🔬 ANALYSE DE PRÉTRAITEMENT - PATIENT ")
print("="*60)

# Récupérer le chemin du premier patient depuis l'inventaire (évite les chemins hardcodés)
from src.O1_preprocessing.loader import get_available_patients
_inv = get_available_patients("../data")

if not _inv['with_both']:
    print("❌ Aucun patient trouvé dans data/raw/ — vérifiez la structure du dossier")
    sys.exit(1)

first_id = _inv['with_both'][0]
patient_path = _inv['vol_paths'][first_id]
use_processed = _inv.get('use_processed', False)
print(f"\n📁 Patient: {patient_path}")
print(f"📂 Mode: {'PROCESSED (.npy)' if use_processed else 'RAW (.dcm)'}")



## 4. Visualisation comparative

### 4.1 Affichage côte à côte des fenêtrages

Comparaison visuelle des différents fenêtrages sur la coupe médiane.
Valide que le prétraitement produit des images exploitables pour la détection.


In [ ]:
# 1. Chargement du volume complet avec métadonnées
print("\n📊 Chargement du volume...")
if use_processed:
    from src.O1_preprocessing.loader import load_npy_volume
    volume, metadata = load_npy_volume(str(patient_path / "volume.npy"))
else:
    from src.O1_preprocessing.loader import load_dicom_volume
    volume, metadata = load_dicom_volume(str(patient_path))

if volume is None:
    print("❌ Erreur de chargement")
    sys.exit(1)

if not use_processed and metadata:
    spacing = metadata.get('spacing', [1.0, 1.0])
    slice_spacing = metadata.get('slice_spacing', 1.0)
    print(f"\n📈 Statistiques du volume:")
    print(f"   Dimensions: {volume.shape} (Z={volume.shape[0]}, Y={volume.shape[1]}, X={volume.shape[2]})")
    print(f"   Résolution spatiale: XY={spacing[0]:.2f}×{spacing[1]:.2f} mm, Z={slice_spacing:.2f} mm")
    print(f"   Volume total approximatif: {volume.shape[0] * slice_spacing * spacing[0] * spacing[1] / 1000:.1f} cm³")
else:
    print(f"\n📈 Statistiques du volume:")
    print(f"   Dimensions: {volume.shape} (Z={volume.shape[0]}, Y={volume.shape[1]}, X={volume.shape[2]})")
    print(f"   Résolution spatiale: N/A (données prétraitées)")
    print(f"   Volume total approximatif: N/A (données prétraitées)")

if use_processed:
    print(f"   Plage normalisée [0,1]: [{volume.min():.3f}, {volume.max():.3f}]")
    
    if volume.min() == volume.max():
        print(f"   ⚠️ Attention : volume constant (toutes les valeurs identiques !)")
        print(f"      Vérifie le preprocessing : le clip [-1000, 400] a-t-il échoué ?")
else:
    print(f"   Plage HU originale: [{volume.min():.1f}, {volume.max():.1f}]")

print(f"   Type de données: {volume.dtype}")



### 4.2 Histogramme des intensités

Distribution des voxels pour valider la cohérence de la normalisation.
Un volume correctement prétraité doit montrer un pic marqué autour de 0 (air) et une longue queue à droite (tissus mous, os).


### Rapport:

Ici, quand j'avais fais le 1er chargement, la `Plage normalisée [0,1]` etait: `[0.715, 0.715]`; un volume constant.
Pour que le résultat soit constant à 0.715, il faudrait que vol soit constant à 400 (car (400 + 1000) / 1400 = 1400/1400 = 1.0, pas 0.715).
Et en verifiant, j'ai vu que le volume DICOM sauvegardé dans `volume.npy` contient des valeurs qui n'ont jamais été converties en HU `(HU = pixel * slope + intercept)`. Le volume est en raw pixel values.

In [ ]:
# 2. Application des différentes fenêtres HU
print("\n🔄 Application des fenêtrages...")

windowed_volumes = {}
if use_processed:
    # Volume déjà normalisé [0, 1] — on l'utilise tel quel
    windowed_volumes['Pulmonaire (embolies)'] = volume
    print("   ✅ Volume déjà normalisé [0,1] (processed) — utilisé directement")
else:
    # Volume en HU — on peut appliquer les fenêtres
    windows_to_test = {
        'Pulmonaire (embolies)': HU_WINDOWS['lung'],
        'Médiastinal (vaisseaux)': HU_WINDOWS['mediastinal'],
        'Osseux': HU_WINDOWS['bone'],
        'Pleine échelle': HU_WINDOWS['full']
    }
    for name, (wmin, wmax) in windows_to_test.items():
        vol_norm = apply_windowing(volume, wmin, wmax, normalize=True)
        windowed_volumes[name] = vol_norm
        print(f"   ✅ {name}: [{wmin}, {wmax}] HU -> normalisé [{vol_norm.min():.3f}, {vol_norm.max():.3f}]")



In [ ]:
# 3. Visualisation comparative des fen\u00eatrages
mid = volume.shape[0] // 2

if use_processed:
    # Mode PROCESSED : volume d\u00e9j\u00e0 normalis\u00e9 [0,1]
    # On montre 6 coupes repr\u00e9sentatives r\u00e9parties sur tout le volume
    indices = [int(volume.shape[0] * r) for r in [0.1, 0.25, 0.4, 0.55, 0.7, 0.85]]
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    for ax, idx in zip(axes, indices):
        im = ax.imshow(volume[idx], cmap='gray', vmin=0, vmax=1)
        ax.set_title(f'Coupe {idx} (normalis\u00e9e [0,1])')
        ax.axis('off')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.suptitle('Coupes du volume pr\u00e9trait\u00e9 (normalis\u00e9 [0,1])', fontsize=14, fontweight='bold')
else:
    # Mode RAW : comparaison des diff\u00e9rents fen\u00eatrages HU
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    im = axes[0].imshow(volume[mid], cmap='gray')
    axes[0].set_title(f'Original (HU) - Coupe {mid}')
    plt.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)
    i = 0
    for i, (name, vol_norm) in enumerate(windowed_volumes.items(), 1):
        if i < len(axes):
            im = axes[i].imshow(vol_norm[mid], cmap='gray')
            axes[i].set_title(f'{name} (normalis\u00e9)')
            plt.colorbar(im, ax=axes[i], fraction=0.046, pad=0.04)
    for j in range(i+1, len(axes)):
        axes[j].axis('off')
    plt.suptitle('Comparaison des fen\u00eatrages Hounsfield', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../figures/preprocessing/comparaison_fenetres.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# CELLULE 4 : Analyse de l'histogramme HU
# Note : en mode PROCESSED, les valeurs sont normalis\u00e9es [0,1]
#        Les axes affichent donc [0,1] et non des HU brutes.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if use_processed:
    # Volume normalis\u00e9 [0,1] : les fen\u00eatres HU se traduisent en fractions
    HU_MIN, HU_MAX = -600, 400  # fen\u00eatre utilis\u00e9e lors du preprocessing
    x_label = 'Valeur normalis\u00e9e [0, 1]  (fen\u00eatre HU [-600, 400])' 
    # Borne pulmonaire : -600 HU \u2192 0.0, +400 HU \u2192 1.0
    v_lung_lo = (-600 - HU_MIN) / (HU_MAX - HU_MIN)  # 0.0
    v_lung_hi = (400  - HU_MIN) / (HU_MAX - HU_MIN)  # 1.0
    # Fen\u00eatre m\u00e9diastinale : 40 \u2192 0.64, 400 \u2192 1.0
    v_med_lo  = (40   - HU_MIN) / (HU_MAX - HU_MIN)  # 0.64
    v_med_hi  = (400  - HU_MIN) / (HU_MAX - HU_MIN)  # 1.0
    # Zone pulmonaire pour histogramme droit
    roi_mask  = (volume > 0.0) & (volume < 1.0)  # tout le volume normalis\u00e9
    zoom_label = 'Valeur normalis\u00e9e [0,1] - Volume complet'
    hist_title = 'Histogramme - Volume normalis\u00e9 [0,1]'
else:
    x_label = 'Unit\u00e9s Hounsfield (HU)'
    v_lung_lo, v_lung_hi = -600, 400  # fen\u00eatre EP retenue [-600, 400]
    v_med_lo,  v_med_hi  = 40,  400
    roi_mask  = (volume > -600) & (volume < 400)
    zoom_label = 'Unit\u00e9s Hounsfield (HU) - Fen\u00eatre pulmonaire'
    hist_title = 'Histogramme - R\u00e9gion pulmonaire [-600, 400] HU'

# Histogramme complet
axes[0].hist(volume.flatten(), bins=200, alpha=0.7, color='steelblue', edgecolor='black', linewidth=0.5)
axes[0].set_xlabel(x_label)
axes[0].set_ylabel('Fr\u00e9quence')
axes[0].set_title('Histogramme des intensit\u00e9s (!\u00e9chelle log)')
axes[0].set_yscale('log')
axes[0].axvline(v_lung_lo, color='red',   linestyle='--', alpha=0.7, label=f'Fen\u00eatre pulmonaire ({v_lung_lo})')
axes[0].axvline(v_lung_hi, color='red',   linestyle='--', alpha=0.7, label=f'Fen\u00eatre pulmonaire ({v_lung_hi})')
axes[0].axvline(v_med_lo,  color='green', linestyle=':',  alpha=0.7, label=f'Fen\u00eatre m\u00e9diastinale ({v_med_lo})')
axes[0].axvline(v_med_hi,  color='green', linestyle=':',  alpha=0.7, label=f'Fen\u00eatre m\u00e9diastinale ({v_med_hi})')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Zoom sur la r\u00e9gion d'int\u00e9r\u00eat
roi_values = volume[roi_mask]
axes[1].hist(roi_values.flatten(), bins=100, alpha=0.7, color='coral', edgecolor='black', linewidth=0.5)
axes[1].set_xlabel(zoom_label)
axes[1].set_ylabel('Fr\u00e9quence')
axes[1].set_title(hist_title)
axes[1].axvline(np.mean(roi_values),   color='red',   linestyle='-',  label=f'Moyenne: {np.mean(roi_values):.3f}')
axes[1].axvline(np.median(roi_values), color='green', linestyle='--', label=f'M\u00e9diane: {np.median(roi_values):.3f}')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/preprocessing/histogramme_hu.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\u2705 Statistiques de la r\u00e9gion d\u0027int\u00e9r\u00eat :')
print(f'   Moyenne : {np.mean(roi_values):.3f}')
print(f'   M\u00e9diane : {np.median(roi_values):.3f}')
print(f'   \u00c9cart-type : {np.std(roi_values):.3f}')


In [ ]:
# ═══════════════════════════════════════════════════
# Distribution des voxels par plage normalisée [0,1]
# ═══════════════════════════════════════════════════
if use_processed:
    t = volume.size
    for lbl, cnt in {
        'Air/vide   [0.00-0.10]': (volume<.10).sum(),
        'Poumon     [0.10-0.40]': ((volume>=.10)&(volume<.40)).sum(),
        'Tissu mou  [0.40-0.65]': ((volume>=.40)&(volume<.65)).sum(),
        'Vaisseaux  [0.65-0.90]': ((volume>=.65)&(volume<.90)).sum(),
        'Os/dense   [0.90-1.00]': (volume>=.90).sum(),
    }.items():
        print(f'{lbl:28s}  {cnt:12,d}  {100*cnt/t:6.2f}%')
else:
    print('Mode RAW — lancez le preprocessing (cellule 3).')


## 5. Vérification du masque GT

### 5.1 Chargement et validation du masque Ground Truth

Chargement du masque GT et vérification de sa cohérence spatiale avec le volume.
Contrôle que les annotations couvrent une zone anatomiquement plausible.

> **Point d'attention :** un masque GT vide (tout à zéro) indique un patient sans embolie confirmée ou une erreur de chargement.


In [ ]:
# ═══════════════════════════════════════════════════
# Vérification masque Ground Truth
# ═══════════════════════════════════════════════════
from src.O1_preprocessing.loader import load_npy_mask
if use_processed:
    gt_mask, _ = load_npy_mask(str(patient_path / 'mask.npy'))
    uv  = np.unique(gt_mask)
    n   = int((gt_mask > 0).sum())
    pct = 100 * n / gt_mask.size
    print(f'Shape  : {gt_mask.shape}  dtype={gt_mask.dtype}')
    print(f'Valeurs uniques  : {uv}')
    print(f'Voxels embolie   : {n:,} ({pct:.4f}%)')
    print('Binaire OK ✅' if set(uv) <= {0,1} else f'Non binaire ⚠️ : {uv}')
else:
    print('Mode RAW — masque GT via load_mask() sur les .mat')


### **Observation :** Histo des intensitées HU (echelle log)

De ce graphe, on vois un 1er pic; à la position `~ -1000 HU`; anatomiquement, cela correspond à **l'air pur** (autour du patient + poumons)

Un 2ème pic qui flote entre la position ``~0 à +100 HU``; cela correspond au Tissus mous(vaisseaux, mediastin, embolies)

Apres cela une chute entre la position `+400 à +3000 HU`; ce qui correspond aux Os dense + contraste iodé + artefacts

### 5.2 Montage des coupes avec superposition GT

Affichage de plusieurs coupes avec le masque GT superposé.
Vérifie visuellement l'alignement entre les annotations et le scan.


In [ ]:
# 5. Visualisation 3D des coupes (montage)
n_cols = 8
n_rows = (volume.shape[0] + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 2.5))
axes = axes.flatten() if n_rows > 1 else [axes]

# Utilisation de la fenêtre pulmonaire pour la visualisation
vol_lung = windowed_volumes['Pulmonaire (embolies)']

for i in range(len(axes)):
    if i < volume.shape[0]:
        axes[i].imshow(vol_lung[i], cmap='gray')
        axes[i].set_title(f'Slice {i}')
        axes[i].axis('off')
    else:
        axes[i].axis('off')

plt.suptitle('Montage des coupes - Fenêtre pulmonaire normalisée', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/preprocessing/montage_coupes.png', dpi=150, bbox_inches='tight')
plt.show()



### 5.3 Alignement automatique des dimensions

Correction automatique si les dimensions du volume et du masque divergent.
Ce cas peut survenir si le prétraitement a recadré le volume.


In [ ]:
if mask is not None:
    # Vérification critique
    if volume.shape != mask.shape:
        print(f"⚠️ ALERT: Shapes différentes!")
        print(f"   Volume: {volume.shape}")
        print(f"   Masque: {mask.shape}")
        print(f"   -> Ajustement automatique")

        min_z = min(volume.shape[0], mask.shape[0])
        volume = volume[:min_z]
        mask = mask[:min_z]

### 5.4 Validation numérique du prétraitement

Vérification programmatique que les valeurs sont dans la plage [0,1] et qu'il n'y a pas de NaN/Inf.


In [ ]:
# ═════════════════════════════════════
#  Validation du Prétraitement
# ═════════════════════════════════════
print("🔍 Validation du prétraitement...")

vol_lung = windowed_volumes['Pulmonaire (embolies)']

# Vérifier que les valeurs sont bien dans [0, 1]
assert vol_lung.min() >= 0, "Erreur: valeurs < 0"
assert vol_lung.max() <= 1, "Erreur: valeurs > 1"
assert vol_lung.dtype == np.float32, "Erreur: mauvais dtype"

print("✅ Toutes les vérifications sont passées !")
print(f"   Plage: [{vol_lung.min():.3f}, {vol_lung.max():.3f}]")
print(f"   Type: {vol_lung.dtype}")

### 5.5 Superposition scan / masque GT

Visualisation finale superposant le scan et le masque GT sur la coupe la plus riche en embolie.


In [ ]:
# ═══════════════════════════════════════════════════════════
# Superposition : scan + fenêtre pulmonaire + masque GT
# ═══════════════════════════════════════════════════════════
from src.O1_preprocessing.loader import load_npy_mask
if use_processed:
    gt_mask, _ = load_npy_mask(str(patient_path / 'mask.npy'))
else:
    from src.O1_preprocessing.loader import load_mask
    gt_mask, _ = load_mask(str(_inv['mask_paths'][first_id]))
if gt_mask is not None:
    emb_per_slice = gt_mask.sum(axis=(1,2))
    best = int(emb_per_slice.argmax()) if emb_per_slice.max() > 0 else volume.shape[0]//2
    vol_d = windowed_volumes.get('Pulmonaire (embolies)', volume)
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(volume[best], cmap='gray')
    axes[0].set_title('Volume normalisé [0,1]' if use_processed else 'HU brut')
    axes[0].axis('off')
    axes[1].imshow(vol_d[best], cmap='gray')
    axes[1].set_title('Fenêtre pulmonaire [-600, 400]')
    axes[1].axis('off')
    axes[2].imshow(vol_d[best], cmap='gray')
    if gt_mask[best].max() > 0: axes[2].imshow(gt_mask[best], cmap='Reds', alpha=0.5)
    axes[2].set_title(f'GT (rouge) — coupe {best}')
    axes[2].axis('off')
    plt.suptitle(f'Patient {first_id}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'../figures/preprocessing/superposition_{first_id}.png', dpi=150)
    plt.show()
    print(f'Coupe {best} — {int(emb_per_slice[best])} voxels embolie')


### Observation — Superposition scan / masque GT

De gauche à droite :
1. **Volume brut** — normalisé [0,1] (processed) ou HU (raw)
2. **Fenêtre pulmonaire** [-600, 400] — met en évidence parenchyme et vaisseaux
3. **Superposition GT** — masque ground truth en rouge sur la coupe la plus riche en embolie
